# 03 — YOLOv8 Fine-Tuning on xView Satellite Dataset

Fine-tune YOLOv8 on the **[xView](http://xviewdataset.org/)** dataset for satellite object detection across 60 classes.

**Run on Google Colab with GPU runtime** (Runtime → Change runtime type → T4 GPU).

**Prerequisites:** You need a [Kaggle](https://www.kaggle.com/) account with API credentials (`~/.kaggle/kaggle.json`).

In [ ]:
!pip install -q ultralytics kaggle

In [ ]:
import os
import json
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

## 1. Setup Kaggle Credentials

Upload your `kaggle.json` or set credentials manually.

In [ ]:
# Option A: Upload kaggle.json in Colab
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

if not (kaggle_dir / 'kaggle.json').exists():
    print('Upload your kaggle.json file:')
    from google.colab import files
    uploaded = files.upload()
    for fn, content in uploaded.items():
        (kaggle_dir / 'kaggle.json').write_bytes(content)
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
    print('Kaggle credentials saved.')
else:
    print('Kaggle credentials found.')

## 2. Download xView Dataset

In [ ]:
DATA_ROOT = Path('xview_dataset')
DATA_ROOT.mkdir(exist_ok=True)

if not (DATA_ROOT / 'train_images').exists():
    print('Downloading xView dataset from Kaggle (~20 GB)...')
    !kaggle datasets download -d fourthbrain/xview-dataset -p {DATA_ROOT} --unzip
    print('Done!')
else:
    print('xView dataset already downloaded.')

# Find the GeoJSON annotations file
geojson_candidates = list(DATA_ROOT.rglob('*.geojson')) + list(DATA_ROOT.rglob('*train*.json'))
print(f'Annotation files found: {[str(p) for p in geojson_candidates]}')

## 3. Define xView Class Mapping (60 classes)

xView uses sparse class IDs (11, 12, 13, ... 94). We map them to contiguous 0-59.

In [ ]:
# xView class ID → name mapping
XVIEW_CLASSES = {
    11: 'Fixed-wing Aircraft', 12: 'Small Aircraft', 13: 'Cargo Plane',
    15: 'Helicopter', 17: 'Passenger Vehicle', 18: 'Small Car',
    19: 'Bus', 20: 'Pickup Truck', 21: 'Utility Truck',
    23: 'Truck', 24: 'Cargo Truck', 25: 'Truck w/Box',
    26: 'Truck Tractor', 27: 'Trailer', 28: 'Truck w/Flatbed',
    29: 'Truck w/Liquid', 32: 'Crane Truck', 33: 'Railway Vehicle',
    34: 'Passenger Car', 35: 'Cargo Car', 36: 'Flat Car',
    37: 'Tank Car', 38: 'Locomotive', 40: 'Maritime Vessel',
    41: 'Motorboat', 42: 'Sailboat', 44: 'Tugboat',
    45: 'Barge', 47: 'Fishing Vessel', 49: 'Ferry',
    50: 'Yacht', 51: 'Container Ship', 52: 'Oil Tanker',
    53: 'Engineering Vehicle', 54: 'Tower Crane', 55: 'Container Crane',
    56: 'Reach Stacker', 57: 'Straddle Carrier', 59: 'Mobile Crane',
    60: 'Dump Truck', 61: 'Haul Truck', 62: 'Scraper/Tractor',
    63: 'Front Loader/Bulldozer', 64: 'Excavator', 65: 'Cement Mixer',
    66: 'Ground Grader', 71: 'Hut/Tent', 72: 'Shed',
    73: 'Building', 74: 'Aircraft Hangar', 76: 'Damaged Building',
    77: 'Facility', 79: 'Construction Site', 83: 'Vehicle Lot',
    84: 'Helipad', 86: 'Storage Tank', 89: 'Shipping Container Lot',
    91: 'Shipping Container', 93: 'Pylon', 94: 'Tower',
}

# Create contiguous mapping
SPARSE_IDS = sorted(XVIEW_CLASSES.keys())
SPARSE_TO_CONTIGUOUS = {sparse_id: idx for idx, sparse_id in enumerate(SPARSE_IDS)}
CONTIGUOUS_NAMES = [XVIEW_CLASSES[sid] for sid in SPARSE_IDS]

print(f'Total classes: {len(CONTIGUOUS_NAMES)}')
for i, name in enumerate(CONTIGUOUS_NAMES):
    print(f'  {i:2d}: {name}')

## 4. Convert GeoJSON Annotations to YOLO Format

In [ ]:
YOLO_DIR = Path('xview_yolo')
YOLO_TRAIN_IMG = YOLO_DIR / 'train' / 'images'
YOLO_TRAIN_LBL = YOLO_DIR / 'train' / 'labels'
YOLO_VAL_IMG = YOLO_DIR / 'val' / 'images'
YOLO_VAL_LBL = YOLO_DIR / 'val' / 'labels'

for d in [YOLO_TRAIN_IMG, YOLO_TRAIN_LBL, YOLO_VAL_IMG, YOLO_VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

def convert_xview_to_yolo(geojson_path, image_dir):
    """Convert xView GeoJSON annotations to YOLO .txt format."""
    with open(geojson_path) as f:
        data = json.load(f)

    # Group annotations by image
    annotations = defaultdict(list)
    skipped = 0

    for feature in data['features']:
        props = feature['properties']
        type_id = props.get('type_id')
        image_id = props.get('image_id', '')

        if type_id not in SPARSE_TO_CONTIGUOUS:
            skipped += 1
            continue

        coords = feature['geometry']['coordinates'][0]  # polygon ring
        xs = [c[0] for c in coords]
        ys = [c[1] for c in coords]
        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

        annotations[image_id].append({
            'class_id': SPARSE_TO_CONTIGUOUS[type_id],
            'xmin': xmin, 'ymin': ymin,
            'xmax': xmax, 'ymax': ymax,
        })

    print(f'Loaded {sum(len(v) for v in annotations.values())} annotations '
          f'across {len(annotations)} images (skipped {skipped} unknown classes)')
    return annotations

# Find and convert annotations
geojson_path = geojson_candidates[0] if geojson_candidates else None
if geojson_path is None:
    raise FileNotFoundError('No GeoJSON annotation file found in xView download.')

print(f'Using annotation file: {geojson_path}')
annotations = convert_xview_to_yolo(geojson_path, DATA_ROOT)

In [ ]:
# Write YOLO label files and symlink/copy images
image_dir = DATA_ROOT / 'train_images'
if not image_dir.exists():
    # Try alternative directory structure
    candidates = list(DATA_ROOT.glob('**/images')) + list(DATA_ROOT.glob('**/train_images*'))
    image_dir = candidates[0] if candidates else DATA_ROOT
    print(f'Using image directory: {image_dir}')

image_names = sorted(annotations.keys())

# 80/20 train/val split
rng = np.random.default_rng(42)
rng.shuffle(image_names)
split_idx = int(0.8 * len(image_names))
train_images = image_names[:split_idx]
val_images = image_names[split_idx:]

def write_yolo_labels(image_list, img_out_dir, lbl_out_dir):
    count = 0
    for img_name in image_list:
        # Find the actual image file
        img_path = image_dir / img_name
        if not img_path.exists():
            # Try common variations
            for candidate in image_dir.glob(f'{Path(img_name).stem}.*'):
                img_path = candidate
                break
        if not img_path.exists():
            continue

        # Get image dimensions
        img = Image.open(img_path)
        w, h = img.size

        # Copy/link image
        dst_img = img_out_dir / img_path.name
        if not dst_img.exists():
            shutil.copy2(img_path, dst_img)

        # Write YOLO labels
        label_path = lbl_out_dir / f'{img_path.stem}.txt'
        with open(label_path, 'w') as f:
            for ann in annotations[img_name]:
                # Convert to normalized x_center, y_center, width, height
                x_center = ((ann['xmin'] + ann['xmax']) / 2) / w
                y_center = ((ann['ymin'] + ann['ymax']) / 2) / h
                bw = (ann['xmax'] - ann['xmin']) / w
                bh = (ann['ymax'] - ann['ymin']) / h

                # Clamp to [0, 1]
                x_center = max(0, min(1, x_center))
                y_center = max(0, min(1, y_center))
                bw = max(0, min(1, bw))
                bh = max(0, min(1, bh))

                if bw > 0 and bh > 0:
                    f.write(f'{ann["class_id"]} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}\n')
        count += 1
    return count

n_train = write_yolo_labels(train_images, YOLO_TRAIN_IMG, YOLO_TRAIN_LBL)
n_val = write_yolo_labels(val_images, YOLO_VAL_IMG, YOLO_VAL_LBL)
print(f'YOLO dataset ready: {n_train} train images, {n_val} val images')

## 5. Create `data.yaml` Config

In [ ]:
import yaml

data_yaml = {
    'path': str(YOLO_DIR.resolve()),
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(CONTIGUOUS_NAMES),
    'names': CONTIGUOUS_NAMES,
}

yaml_path = YOLO_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f'data.yaml written to {yaml_path}')
print(f'Classes: {len(CONTIGUOUS_NAMES)}')

## 6. Fine-Tune YOLOv8

In [ ]:
# Load pretrained YOLOv8-nano
model = YOLO('yolov8n.pt')
print(model.info())

In [ ]:
# Train
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='satellite_detection',
    name='yolov8_xview',
    patience=10,
    save=True,
    plots=True,
)

## 7. Validation Results

In [ ]:
# Load best weights and validate
best_weights = Path('satellite_detection/yolov8_xview/weights/best.pt')
best_model = YOLO(str(best_weights))

metrics = best_model.val(data=str(yaml_path))
print(f'\nmAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')

## 8. Sample Detection Visualizations

In [ ]:
# Run inference on validation images
val_images_list = sorted(YOLO_VAL_IMG.glob('*.*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()

for i, img_path in enumerate(val_images_list):
    results = best_model.predict(source=str(img_path), conf=0.25, verbose=False)
    annotated = results[0].plot()  # BGR numpy array with boxes drawn
    annotated = annotated[:, :, ::-1]  # BGR → RGB

    axes[i].imshow(annotated)
    n_det = len(results[0].boxes)
    axes[i].set_title(f'{img_path.name} — {n_det} detections', fontsize=10)
    axes[i].axis('off')

plt.suptitle('YOLOv8 Detections on xView Validation Images', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Export Weights

In [ ]:
# Copy best weights to a standard name
export_path = Path('yolov8_satellite.pt')
shutil.copy2(best_weights, export_path)
print(f'Weights saved as {export_path}')
print(f'File size: {export_path.stat().st_size / 1e6:.1f} MB')

# Uncomment to download from Colab:
# from google.colab import files
# files.download('yolov8_satellite.pt')